In [17]:
import requests
import json
import base64
import time
import os
import pandas as pd
from IPython.display import display, Markdown

# ---------- 1. Model feature detection ----------
def get_model_features(model_name):
    """
    Determine if model has thinking/reasoning and vision.
    1. Try to load CSV and look up 'Input' column.
    2. If not found, use name heuristics.
    Returns (has_thinking, has_vision)
    """
    # Try to read combined CSV (if you have it)
    csv_path = r"E:\llm\Model\csv\all_models_combined.csv"
    input_type = None
    if os.path.exists(csv_path):
        try:
            df = pd.read_csv(csv_path)
            row = df[df['Name'] == model_name]
            if not row.empty:
                input_type = row.iloc[0].get('Input', '')
        except:
            pass

    # Vision detection from CSV if available
    if input_type is not None:
        has_vision = 'Image' in str(input_type)
    else:
        # Heuristic based on name
        low = model_name.lower()
        vision_patterns = ['vl', 'vision', 'llava', 'llama3.2-vision', 'llama4',
                           'gemma3', 'gemma4', 'ministral', 'mistral-small3.1',
                           'mistral-medium-3.5', 'medgemma', 'qwen2.5vl', 'qwen3-vl']
        has_vision = any(p in low for p in vision_patterns)

    # Thinking detection
    low = model_name.lower()
    thinking_patterns = ['thinking', 'reasoning', 'deepseek-r1', 'phi4-reasoning',
                         'phi4-mini-reasoning', 'nemotron-3.5-lightning',
                         'nemotron-3-nano', 'nemotron-3-super']
    has_thinking = any(p in low for p in thinking_patterns)
    # Some models may output reasoning without "thinking" in name (e.g., deepseek-r1)
    if 'deepseek-r1' in low:
        has_thinking = True

    return has_thinking, has_vision

# ---------- 2. Image helpers ----------
def image_to_base64_from_path(image_path):
    with open(image_path, "rb") as f:
        return base64.b64encode(f.read()).decode("utf-8")

def image_to_base64_from_url(image_url):
    resp = requests.get(image_url, timeout=30)
    resp.raise_for_status()
    return base64.b64encode(resp.content).decode("utf-8")

def image_to_base64(image_input):
    if image_input.startswith(("http://", "https://")):
        return image_to_base64_from_url(image_input)
    else:
        return image_to_base64_from_path(image_input)

# ---------- 3. Main chat function ----------
def chat_aware(pno,prompt, model="qwen3:4b", image_input=None):
    """
    Chat with automatic handling of thinking/reasoning and vision.
    - If model supports vision and no image_input given, ask user for path/URL.
    - Displays model features in the output.
    """
    has_thinking, has_vision = get_model_features(model)

    # If vision model and no image provided, ask interactively
    # if has_vision and image_input is None:
    #     print(f"🔍 Model '{model}' supports images.")
    #     img = input("Enter image path or URL (or press Enter to skip): ").strip()
    #     if img:
    #         image_input = img

    # Build message
    message = {"role": "user", "content": prompt}
    # if image_input:
    #     try:
    #         image_b64 = image_to_base64(image_input)
    #         message["images"] = [image_b64]
    #     except Exception as e:
    #         print(f"❌ Error loading image: {e}")
    #         return

    payload = {
        "model": model,
        "messages": [message],
        "stream": True,
    }

    resp = requests.post(
        "http://localhost:11434/api/chat",
        json=payload,
        stream=True,
        timeout=300,
    )
    resp.raise_for_status()

    # Timers
    start_time = time.time()
    first_thinking_time = None
    first_content_time = None
    thinking_text = ""
    content_text = ""

    # Create display handle
    handle = display(Markdown(""), display_id=True)

    # Build feature header
    features = []
    if has_thinking:
        features.append("🧠 **Thinking**")
    if has_vision:
        features.append("👁️ **Vision**")
    features_str = ", ".join(features) if features else "None"
    header = f"### Model: {model}\n**Features:** {features_str}\n\n"

    for line in resp.iter_lines(decode_unicode=True):
        if not line:
            continue
        data = json.loads(line)
        msg = data.get("message", {})

        # Thinking
        if "thinking" in msg and msg["thinking"]:
            thinking_text += msg["thinking"]
            if first_thinking_time is None:
                first_thinking_time = time.time()

        # Content
        if "content" in msg and msg["content"]:
            content_text += msg["content"]
            if first_content_time is None:
                first_content_time = time.time()

        # Build markdown
        md = header
        md += f"## 💎 Page {pno}"
        md += f"\n----\n"
        md +=f"## 💖💟 Prompt ♻️🚷🚯🚱🔞🚭☣️☢️"
        md += f"\n----\n"
        md += f"{prompt}"
        md += f"\n----\n"
        if thinking_text:
            md += f"## 🤔 Thinking\n\n{thinking_text}\n\n---\n\n"
        md += f"## 💬 Answer\n\n{content_text}"

        handle.update(Markdown(md))

        if data.get("done"):
            break

    total_time = time.time() - start_time

    # Final timings
    timings = ""
    if first_thinking_time:
        timings += f"**TTFT (thinking):** {first_thinking_time - start_time:.3f}s  \n"
    if first_content_time:
        timings += f"**TTFT (content):** {first_content_time - start_time:.3f}s  \n"
    timings += f"**Total time:** {total_time:.3f}s"

    handle.update(Markdown(md + "\n---\n" + timings))

    return thinking_text, content_text

In [12]:
FILE_NAME = "../../NNDesign.pdf"
FILE_NAME_PAPER = "../../2608.02980v1.pdf"


import pymupdf
import os

Folder1 = "border"

os.makedirs(Folder1, exist_ok=True)

doc = pymupdf.open(FILE_NAME_PAPER)

print(f"Total pages: {doc.page_count}")

def choose(pno):
    page = doc[pno]
    blocks = page.get_text("blocks")
    print(f"Page {pno+1}: {len(blocks)} block(s)")
    return blocks
    #for block in blocks:
        # Tuple: (x0, y0, x1, y1, text, block_no, block_type)
        #rect = pymupdf.Rect(block[0], block[1], block[2], block[3])
        #print(block[4])
        #return block
    #return blocks
        

Total pages: 21


In [13]:
def choose(pno,model="granite4:350m-h"):
    page = doc[pno]
    blocks = page.get_text("blocks")
    #print(f"Page {pno+1}: {len(blocks)} block(s)")
    #return blocks
    for block in blocks:
        # Tuple: (x0, y0, x1, y1, text, block_no, block_type)
        #rect = pymupdf.Rect(block[0], block[1], block[2], block[3])
        #print(block[4])
        #return block
        chat_aware(pno,block[4],model)
    pass


In [ ]:
import pymupdf
import os

Folder1 = "border"

os.makedirs(Folder1, exist_ok=True)

doc = pymupdf.open(FILE_NAME_PAPER)

print(f"Total pages: {doc.page_count}")

for pno in range(doc.page_count):
 choose(pno,model="granite4:3b-h" \
 "")

Total pages: 21


### Model: granite4:3b-h
**Features:** None

## 💎 Page 0
----
## 💖💟 Prompt ♻️🚷🚯🚱🔞🚭☣️☢️
----
Qwen-3D: A Generalist 3D Vision-Language Model for Spatial Understanding

----
## 💬 Answer

Qwen-3D is a significant advancement in the field of artificial intelligence, particularly in the domain of vision-language models. It is designed to understand and process three-dimensional spatial data, a capability that's crucial for a wide range of applications, from robotics and autonomous vehicles to virtual reality and augmented reality. 

Qwen-3D is a Generalist 3D Vision-Language Model, which suggests it has been trained on a broad range of datasets that encompass visual and textual information. This "generalist" approach allows the model to effectively interpret and understand complex spatial relationships in 3D environments, as well as to interpret textual information about these environments. 

The "3D" in the name indicates that Qwen-3D is capable of processing data in three dimensions, which is a major step forward from traditional 2D models. This ability allows it to accurately interpret depth and spatial relationships, which is crucial for tasks like object recognition, navigation, and interaction in a three-dimensional space. 

The "Vision-Language Model" part of the name refers to the model's dual focus on visual and textual data. It combines the best of both worlds - visual recognition (understanding what it sees) and language processing (understanding what it's told). This dual capability makes it extremely versatile, as it can interpret both what it observes in its environment and what it is explicitly told about that environment. 

In practical terms, Qwen-3D can be used in a variety of scenarios. For example, in robotics, it could allow a robot to understand its environment by both observing the physical space around it and understanding verbal commands or descriptions. In virtual reality or augmented reality, it could help create more immersive experiences by accurately interpreting the spatial relationships within the virtual environment and understanding any textual descriptions given. 

In conclusion, Qwen-3D represents a significant leap forward in the field of 3D vision-language models. Its ability to understand and interact with three-dimensional environments using both visual and textual data opens up new possibilities for a wide range of applications, from robotics and autonomous vehicles to virtual and augmented reality. As with all AI technologies, its development and use must be guided by ethical considerations to ensure it benefits society as a whole.
---
**TTFT (content):** 0.000s  
**Total time:** 186.055s

### Model: granite4:3b-h
**Features:** None

## 💎 Page 0
----
## 💖💟 Prompt ♻️🚷🚯🚱🔞🚭☣️☢️
----
Lucy Lin†, Ayush Jain†, Yifan Liu, Katerina Fragkiadaki

----
## 💬 Answer

It appears you've listed four names. Could you please provide more details on what information you need about these individuals? They could be colleagues, authors, or participants in a project or event.
---
**TTFT (content):** 0.001s  
**Total time:** 15.852s

### Model: granite4:3b-h
**Features:** None

## 💎 Page 0
----
## 💖💟 Prompt ♻️🚷🚯🚱🔞🚭☣️☢️
----
Carnegie Mellon University

----
## 💬 Answer

Carnegie Mellon University (CMU) is a private research university located in Pittsburgh, Pennsylvania, USA. It was established in 1900 as the Carnegie Institute of Technology and became Carnegie Mellon University in 1967 after merging with the Mellon Institute of Industrial Research. CMU is renowned for its programs in computer science, engineering, and the arts. The university is consistently ranked among the top universities in the world, particularly in the fields of robotics, artificial intelligence, and computer science. Its faculty includes numerous Nobel laureates and Turing Award winners. CMU is also known for its strong emphasis on interdisciplinary research and innovation.
---
**TTFT (content):** 0.002s  
**Total time:** 58.138s

### Model: granite4:3b-h
**Features:** None

## 💎 Page 0
----
## 💖💟 Prompt ♻️🚷🚯🚱🔞🚭☣️☢️
----
{lucylin,ayushj2,yifanliu,kfragki2}@andrew.cmu.edu

----
## 💬 Answer

The provided information seems to be an email address list for a group of individuals associated with Carnegie Mellon University.
---
**TTFT (content):** 0.002s  
**Total time:** 8.635s

...

### Model: granite4:3b-h
**Features:** None

## 💎 Page 7
----
## 💖💟 Prompt ♻️🚷🚯🚱🔞🚭☣️☢️
----
4.5. 2D Vision-Language tasks

----
## 💬 Answer

2D Vision-Language tasks refer to a class of problems that combine visual perception with language understanding. These tasks often involve interpreting images and associating them with textual descriptions. Here are some key aspects of 2D Vision-Language tasks:

1. **Image Understanding**: The first step in these tasks involves processing the visual data from images. This could be anything from recognizing objects, understanding scenes, or even detecting anomalies.

2. **Natural Language Processing (NLP)**: After understanding the visual aspects, the task then involves interpreting textual information. This could be generating a textual description of an image, answering questions based on an image, or even translating the image content into another language.

3. **Applications**: 2D Vision-Language tasks have numerous applications. For instance, in autonomous vehicles, these tasks can help in understanding the surroundings (like detecting obstacles) and interpreting textual data (like traffic signs or signals). In medical imaging, they can assist in interpreting radiological images and associating them with clinical notes.

4. **Techniques**: Various techniques are used to tackle these tasks. Convolutional Neural Networks (CNNs) are commonly used for image understanding. For the language processing part, recurrent neural networks (RNNs), Long Short-Term Memory networks (LSTMs), or Transformer models like BERT (Bidirectional Encoder Representations from Transformers) are often used.

5. **Evaluation Metrics**: The performance of these tasks is typically evaluated using metrics like accuracy (for classification tasks), precision, recall, F1-score (for multi-label classification tasks), and BLEU (Bilingual Evaluation Understudy) score (for tasks like machine translation).

6. **Challenges**: Despite their potential, 2D Vision-Language tasks also pose several challenges. These include dealing with ambiguity in images and text, handling out-of-vocabulary words, and ensuring robustness to variations in image and language styles.

In conclusion, 2D Vision-Language tasks represent a significant intersection of computer vision and natural language processing, with wide-ranging applications and ongoing research to improve their efficiency and effectiveness.
---
**TTFT (content):** 0.001s  
**Total time:** 139.083s

### Model: granite4:3b-h
**Features:** None

## 💎 Page 7
----
## 💖💟 Prompt ♻️🚷🚯🚱🔞🚭☣️☢️
----
To prevent degradation of the base model’s strong 2D ca-
pabilities, we co-train Qwen-3D on 2D datasets, including
the RefCOCO family [27] and LLaVA-Instruct-150k [35].
Tab. 4 evaluates Qwen-3D against its pre-trained base
model (Qwen2.5-VL 3B [5]) on RefCOCO 2D grounding
benchmarks and the held-out RealWorldQA [48] dataset.
As in our 3D setup, Qwen-3D predicts segmentation masks
which we convert to bounding boxes for evaluation. Results
show that Qwen-3D preserves the strong 2D capabilities of
Qwen2.5-VL despite additional 3D fine-tuning, indicating
that geometry-aware reasoning can be added without sacri-
ficing the general multimodal capabilities of the base model.

----
## 💬 Answer

The text you've provided seems to be a summary or a part of a research paper or technical report. It discusses a method to prevent degradation in the strong 2D capabilities of a base model (Qwen2.5-VL 3B) during the addition of 3D fine-tuning. This is achieved by co-training the 3D model (Qwen-3D) on 2D datasets like RefCOCO and LLaVA-Instruct-150k. 

The performance of the modified model, Qwen-3D, is then evaluated against its base model on RefCOCO 2D grounding benchmarks and the RealWorldQA dataset. The results indicate that Qwen-3D maintains the strong 2D capabilities of Qwen2.5-VL, even after undergoing additional 3D fine-tuning. This suggests that geometry-aware reasoning can be successfully incorporated into the model without compromising its general multimodal capabilities.

In simpler terms, the research is about enhancing a 3D model's abilities without losing its existing 2D proficiency. The model (Qwen-3D) is tested and found to retain its strong performance in 2D tasks even after being trained on 3D data. This suggests a promising approach to developing models that can effectively handle tasks in multiple dimensions.
---
**TTFT (content):** 0.001s  
**Total time:** 89.851s

### Model: granite4:3b-h
**Features:** None

## 💎 Page 7
----
## 💖💟 Prompt ♻️🚷🚯🚱🔞🚭☣️☢️
----
For additional visualizations of Qwen-3D, ablations on
pre-training retention, and results on text-only tasks, please
refer to the appendix.

----
## 💬 Answer

Thank you for your request. I have noted your request for additional visualizations of Qwen-3D, ablations on pre-training retention, and results on text-only tasks. These resources are provided in the appendix as per your instructions. If you have any other inquiries or require further assistance, please do not hesitate to ask.
---
**TTFT (content):** 0.001s  
**Total time:** 22.178s

### Model: granite4:3b-h
**Features:** None

## 💎 Page 7
----
## 💖💟 Prompt ♻️🚷🚯🚱🔞🚭☣️☢️
----
4.6. Additional Analysis and Ablations

----
## 💬 Answer

Certainly! "Additional Analysis and Ablations" is typically a section in academic or research papers, particularly in fields like computer science, engineering, or experimental sciences. It is used to further evaluate the results of the study, examine the impact of certain variables or parameters, and assess the robustness of the findings. Here's a detailed explanation of what this section usually entails:

### 1. **Introduction to Additional Analysis**

**Purpose:**  
The primary purpose of this section is to provide deeper insights into the data obtained from the study. It may include exploring new facets of the data, validating the results with different methodologies, or testing the study's hypotheses in various scenarios.

**Common Techniques:**  
- **Statistical Tests:** To verify the significance of the findings. This can include t-tests, ANOVA, chi-square tests, etc., depending on the type of data and the hypothesis being tested.
- **Regression Analysis:** To understand the relationship between variables. This can help in identifying which variables have a significant impact on the outcome.
- **Sensitivity Analysis:** To determine how different values of an independent variable affect a dependent variable under a given set of assumptions. This is crucial for understanding the robustness of the model or hypothesis.
- **Cross-validation:** To assess how the results of a study generalize to an independent dataset. This is especially important in predictive modeling to avoid overfitting.

### 2. **Ablations (or “Ablation Studies”)**

**Definition:**  
Ablation studies are a method of comparative analysis where one or more components of a system are

: 